In [ ]:
import torch
import monai
import matplotlib.pyplot as plt
import numpy as np
from monai.data import Dataset, DataLoader
from monai.utils import set_determinism
from monai.networks.nets import ViT, UNETR
from sklearn.model_selection import train_test_split

In [ ]:
import sys
import os

parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)

from src.data.synthetic import create_synthetic_dataset, TransformSequence, create_synthetic_time_3d
from src.data.utils import (
    pad_sequence_collate_fn,
    get_train_transform_synthetic,
    get_val_transform_synthetic
)
from src.networks.t_unetr import TemporalUNETR
from src.train.train_ops import train

In [ ]:
# Set seed
set_determinism(42)

X = create_synthetic_dataset(
    num_patients=10
)

X_train, X_val = train_test_split(X, test_size=0.2)

In [ ]:
max_len = 0
indx = 0
for i, x in enumerate(X_train):
    if max_len < len(x["images"]):
        max_len = len(x["images"])
        indx = i
print(f"max length is {max_len} on index {indx}")

In [ ]:
images, labels, dates = X_train[3]["images"], X_train[3]["labels"], X_train[3]["dates"]

In [ ]:
def find_tumor_slices(label_volume):
    """
    Find optimal slice positions to visualize the tumor based on the label/mask.
    
    Parameters:
    -----------
    label_volume : np.ndarray
        3D or 4D binary label/mask volume where tumor pixels are > 0
        If 4D, assumes shape (C, H, W, D) and uses first channel
    
    Returns:
    --------
    dict : Dictionary containing slice information
        - 'centroid': (x, y, z) centroid coordinates  
        - 'slices': optimal slice positions for each axis
        - 'max_area_slices': slices with maximum tumor area
        - 'bounding_box': min/max coordinates of tumor
    """
    # Handle channel dimension if present
    if label_volume.ndim == 4:
        # Assume shape is (C, H, W, D) and use first channel
        label_3d = label_volume[0]
        print(f"Detected 4D label volume with shape {label_volume.shape}, using first channel")
    elif label_volume.ndim == 3:
        label_3d = label_volume
    else:
        raise ValueError(f"Expected 3D or 4D label volume, got shape {label_volume.shape}")
    
    # Find all tumor voxels
    tumor_coords = np.where(label_3d > 0)
    
    if len(tumor_coords[0]) == 0:
        # No tumor found, return middle slices
        h, w, d = label_3d.shape
        return {
            'centroid': (h//2, w//2, d//2),
            'slices': {'axial': h//2, 'sagittal': w//2, 'coronal': d//2},
            'max_area_slices': {'axial': h//2, 'sagittal': w//2, 'coronal': d//2},
            'bounding_box': None
        }
    
    # Calculate centroid (center of mass)
    centroid_x = np.mean(tumor_coords[0])
    centroid_y = np.mean(tumor_coords[1]) 
    centroid_z = np.mean(tumor_coords[2])
    centroid = (int(np.round(centroid_x)), int(np.round(centroid_y)), int(np.round(centroid_z)))
    
    # Calculate bounding box
    min_x, max_x = np.min(tumor_coords[0]), np.max(tumor_coords[0])
    min_y, max_y = np.min(tumor_coords[1]), np.max(tumor_coords[1])
    min_z, max_z = np.min(tumor_coords[2]), np.max(tumor_coords[2])
    bounding_box = {
        'x': (min_x, max_x), 'y': (min_y, max_y), 'z': (min_z, max_z)
    }
    
    # Find slices with maximum tumor area for each orientation
    max_area_slices = {}
    
    # Axial (XY plane) - sum across height axis
    axial_areas = np.sum(label_3d, axis=(1, 2))
    max_area_slices['axial'] = np.argmax(axial_areas)
    
    # Sagittal (YZ plane) - sum across width axis  
    sagittal_areas = np.sum(label_3d, axis=(0, 2))
    max_area_slices['sagittal'] = np.argmax(sagittal_areas)
    
    # Coronal (XZ plane) - sum across depth axis
    coronal_areas = np.sum(label_3d, axis=(0, 1))
    max_area_slices['coronal'] = np.argmax(coronal_areas)
    
    # Centroid-based slices
    centroid_slices = {
        'axial': centroid[0],
        'sagittal': centroid[1], 
        'coronal': centroid[2]
    }
    
    return {
        'centroid': centroid,
        'slices': centroid_slices,
        'max_area_slices': max_area_slices,
        'bounding_box': bounding_box
    }

In [ ]:
height, width, depth = 394, 466, 378

# Create comprehensive visualization
fig = plt.figure(figsize=(20, 16))

tumor_info = find_tumor_slices(labels[0])

# Show different slice orientations across time
slice_positions = tumor_info['max_area_slices']  # Use slices with maximum tumor area
    

for row, (orientation, slice_pos) in enumerate(slice_positions.items()):
    for col, (time_idx, date) in enumerate(zip(range(len(dates)), dates)):
        ax = plt.subplot(3, len(dates), row * len(dates) + col + 1)
        
        current_image = images[time_idx][0]

        # Extract the appropriate slice based on orientation
        if orientation == 'axial':
            slice_data = current_image[slice_pos, :, :]
            xlabel, ylabel = 'Width', 'Depth'
        elif orientation == 'sagittal':
            slice_data = current_image[:, slice_pos, :]
            xlabel, ylabel = 'Height', 'Depth'
        else:  # coronal
            slice_data = current_image[:, :, slice_pos]
            xlabel, ylabel = 'Height', 'Width'
        
        # Display the slice
        im = ax.imshow(slice_data, cmap='gray', origin='lower')
        ax.set_title(f'{orientation.title()} - Day {date}', fontsize=10)
        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        
        # Add colorbar for first image of each row
        if col == 0:
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.suptitle('Synthetic 3D MRI Data Evolution Over Time\n(Showing Different Slice Orientations)', 
                fontsize=16, y=0.98)

# Create a second figure showing the labels/masks
fig2 = plt.figure(figsize=(15, 5))

for col, (time_idx, date) in enumerate(zip(range(len(dates)), dates)):
    ax = plt.subplot(1, len(dates), col + 1)
    
    # Show axial slice of the label mask
    current_label = labels[time_idx][0]
    optimal_slice = slice_positions['axial']
    label_slice = current_label[optimal_slice, :, :]
    im = ax.imshow(label_slice, cmap='jet', origin='lower')
    ax.set_title(f'Label Mask - Day {date}', fontsize=12)
    ax.set_xlabel('Width', fontsize=10)
    ax.set_ylabel('Depth', fontsize=10)
    
    if col == 0:
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.suptitle('Segmentation Labels Over Time (Axial View)', fontsize=14, y=1.02)

# Create a third figure showing growth analysis
fig3, axes = plt.subplots(2, 2, figsize=(12, 10))

# Calculate volume over time
volumes = [np.sum(label > 0) for label in labels]
mean_intensities = [np.mean(img[img > 0]) for img in images]
max_intensities = [np.max(img) for img in images]

# Plot volume growth
axes[0, 0].plot(dates, volumes, 'b-o', linewidth=2, markersize=6)
axes[0, 0].set_title('Volume Growth Over Time')
axes[0, 0].set_xlabel('Days')
axes[0, 0].set_ylabel('Volume (voxels)')
axes[0, 0].grid(True, alpha=0.3)

# Plot mean intensity
axes[0, 1].plot(dates, mean_intensities, 'r-s', linewidth=2, markersize=6)
axes[0, 1].set_title('Mean Intensity Over Time')
axes[0, 1].set_xlabel('Days')
axes[0, 1].set_ylabel('Mean Intensity')
axes[0, 1].grid(True, alpha=0.3)

# Plot max intensity
axes[1, 0].plot(dates, max_intensities, 'g-^', linewidth=2, markersize=6)
axes[1, 0].set_title('Maximum Intensity Over Time')
axes[1, 0].set_xlabel('Days')
axes[1, 0].set_ylabel('Max Intensity')
axes[1, 0].grid(True, alpha=0.3)

# Show intensity histogram for last time point
axes[1, 1].hist(images[-1].flatten(), bins=50, alpha=0.7, color='purple')
axes[1, 1].set_title('Intensity Distribution (Final Time Point)')
axes[1, 1].set_xlabel('Intensity')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Quantitative Analysis of Synthetic MRI Data', fontsize=14, y=1.02)

plt.show()

In [ ]:
X_train[0]["images"][0].shape

In [ ]:
spatial_size = (128, 128, 128)

# Create transform
train_transforms = get_train_transform_synthetic(spatial_size)
val_transforms = get_val_transform_synthetic(spatial_size)

In [ ]:
# Create MONAI dataset
transforms = TransformSequence(keys=["images", "labels", "dates"], spatial_transforms=train_transforms)
train_dataset = Dataset(data=X_train, transform=transforms)
transforms = TransformSequence(keys=["images", "labels", "dates"], spatial_transforms=val_transforms)
val_dataset = Dataset(data=X_val, transform=transforms)

In [ ]:
print(train_dataset[0]["images"].shape)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    collate_fn=pad_sequence_collate_fn
)
val_loader = DataLoader(
    train_dataset,
    batch_size=2,
    collate_fn=pad_sequence_collate_fn
)

# TemporalUNETR

Building and testing the TemporalUNETR

In [ ]:
net = TemporalUNETR(
    in_channels=1,
    out_channels=1,
    img_size=(128, 128, 128),
    patch_size=16
)

In [ ]:
some_output = train(net, train_loader, val_loader, max_epochs=2, initial_lr=1e-3)